[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-02-drivers-execution.ipynb#scrollTo=aa000001)

---
# Day 2 · Drivers & Graph Execution
**certified-journeys / hamilton-certified** · Day 2 · Core Mechanics

> **Goal for today:** Master the Driver API — how `Builder` works, the difference between `execute()` and `raw_execute()`, overriding nodes, and visualizing execution subgraphs.

In [ ]:
%pip install -q sf-hamilton

## Step 1 · The Driver Builder Pattern

Hamilton uses a fluent Builder to configure the Driver before building it:

```python
from hamilton import driver

dr = (
    driver.Builder()
    .with_modules(module_a, module_b)   # add function modules
    .with_config({'env': 'dev'})        # optional config dict
    .build()
)
```

| Builder method | Purpose |
|---|---|
| `.with_modules(*mods)` | Register function modules |
| `.with_config(dict)` | Supply config for @config.when decorators |
| `.with_adapters(*adapters)` | Add lifecycle hooks or custom executors |
| `.build()` | Validate and return the Driver |

The build step **validates the graph** — it will raise immediately if any function has a parameter that has no upstream producer.

In [ ]:
import sys, types, pandas as pd
from hamilton import driver

# Define a small pipeline module
def raw_age(age: pd.Series) -> pd.Series:
    """Input: raw age values."""
    return age

def age_clamped(raw_age: pd.Series) -> pd.Series:
    """Clamp age to [0, 120]."""
    return raw_age.clip(lower=0, upper=120)

def age_bucket(age_clamped: pd.Series) -> pd.Series:
    """Bucket age into: young(<30), mid(30-60), senior(60+)."""
    return pd.cut(age_clamped, bins=[0,30,60,120], labels=['young','mid','senior'])

def age_is_senior(age_bucket: pd.Series) -> pd.Series:
    """Boolean: True if bucket is senior."""
    return age_bucket == 'senior'

# Package as a module
age_module = types.ModuleType('age_module')
for fn in [raw_age, age_clamped, age_bucket, age_is_senior]:
    setattr(age_module, fn.__name__, fn)
sys.modules['age_module'] = age_module

# Build with the Builder pattern
dr = driver.Builder().with_modules(age_module).build()
print('Driver built successfully')
print('Available nodes:', [v.name for v in dr.list_available_variables()])

### What just happened?
- **`.with_modules(age_module)`** tells Hamilton to discover all functions in that module.
- **`.build()`** validates the graph — `raw_age` is an input node (its parameter `age` has no upstream producer, so it must be supplied as an input).
- **`list_available_variables()`** lists every node Hamilton found — this is your graph inventory.

## Step 2 · `execute()` vs `raw_execute()`

| Method | Returns | When to use |
|---|---|---|
| `dr.execute(outputs, inputs)` | `dict[str, Any]` — only the requested outputs | Production, feature collection |
| `dr.raw_execute(outputs, inputs)` | `dict[str, Any]` — same shape, but bypasses output adapter | Debugging, when you need the raw Python objects |
| `dr.execute(outputs, inputs, overrides)` | `dict[str, Any]` | When you want to inject a value for a mid-graph node |

**Key difference:** `execute()` passes results through any registered output adapters (e.g., `PandasDataFrameResult`). `raw_execute()` always returns plain Python/pandas objects regardless of adapters.

In [ ]:
inputs = {'age': pd.Series([25.0, 45.0, 70.0, -5.0, 150.0])}

# execute() — request specific outputs
result = dr.execute(['age_clamped', 'age_bucket', 'age_is_senior'], inputs=inputs)
print('execute() result type:', type(result))
print('Keys:', list(result.keys()))
print('\nage_clamped:\n', result['age_clamped'].values)
print('\nage_bucket:\n', result['age_bucket'].values)
print('\nage_is_senior:\n', result['age_is_senior'].values)

In [ ]:
# overrides — inject a precomputed value for age_clamped, skipping that node
# Useful in tests: override upstream to isolate a downstream function
precomputed_clamped = pd.Series([25.0, 45.0, 70.0, 0.0, 120.0])
result_override = dr.execute(
    ['age_bucket', 'age_is_senior'],
    inputs=inputs,
    overrides={'age_clamped': precomputed_clamped}  # bypasses raw_age + age_clamped
)
print('Override result - age_bucket:', result_override['age_bucket'].values)

### What just happened?
- **`execute()`** with specific output names only computes what's needed — Hamilton prunes the graph automatically.
- **`overrides`** injects a pre-computed value into any node, skipping everything upstream. This is extremely useful in tests: provide a known mid-graph value and only test the downstream logic.
- Both `-5.0` and `150.0` ages were clamped by `age_clamped` before reaching `age_bucket`.

## Step 3 · Collecting All Outputs as a DataFrame

A common pattern: collect every feature into a single DataFrame using `PandasDataFrameResult` adapter.

```python
from hamilton.plugins import h_pandas

dr = (
    driver.Builder()
    .with_modules(my_module)
    .with_adapters(h_pandas.PandasDataFrameResult())
    .build()
)
df = dr.execute(['feature_a', 'feature_b', 'feature_c'], inputs=inputs)
# df is now a pd.DataFrame, not a dict
```

Without the adapter, `execute()` returns a `dict`. With `PandasDataFrameResult`, it returns a `pd.DataFrame` with one column per requested output.

In [ ]:
from hamilton.plugins import h_pandas

dr_df = (
    driver.Builder()
    .with_modules(age_module)
    .with_adapters(h_pandas.PandasDataFrameResult())
    .build()
)

df_result = dr_df.execute(
    ['age_clamped', 'age_is_senior'],
    inputs=inputs
)
print(type(df_result))
print(df_result)

### What just happened?
- **`PandasDataFrameResult`** is an output adapter that collects Series outputs into a DataFrame.
- The Driver is otherwise identical — adapters are a cross-cutting concern, not part of your functions.
- This pattern is the most common in feature engineering: run the pipeline, get a feature DataFrame back.

## Step 4 · Inspecting the Graph Programmatically

Beyond visualization, the Driver exposes graph metadata:

```python
dr.list_available_variables()   # all nodes: name + type
dr.what_is_upstream_of('node')  # all ancestors
dr.what_is_downstream_of('node')# all descendants
```

Use these to build documentation, lineage reports, or validation scripts.

In [ ]:
# Inspect the graph structure programmatically
print('=== All nodes ===')
for v in dr.list_available_variables():
    print(f'  {v.name}: {v.type}')

print('\n=== Upstream of age_is_senior ===')
upstream = dr.what_is_upstream_of('age_is_senior')
print([n.name for n in upstream])

print('\n=== Downstream of age_clamped ===')
downstream = dr.what_is_downstream_of('age_clamped')
print([n.name for n in downstream])

### What just happened?
- **`what_is_upstream_of`** returns all ancestors — everything that must execute before a given node.
- **`what_is_downstream_of`** returns all descendants — everything that will be affected if a node changes.
- These are the building blocks for impact analysis: "if I change `age_clamped`, what outputs change?"

In [ ]:
# Challenge: add a new node `age_decade` that maps age_clamped to the decade (0, 10, 20, ...)
# then rebuild the Driver and verify it appears in list_available_variables()
# Also: check what is upstream of age_decade

# def age_decade(age_clamped: pd.Series) -> pd.Series:
#     ...

print('Add age_decade to age_module and rebuild the driver!')

---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| Builder pattern | `.with_modules()` → `.with_config()` → `.with_adapters()` → `.build()` |
| `.build()` validates | Raises immediately if any parameter has no producer |
| `execute()` | Returns dict (or DataFrame with adapter); prunes graph to minimum needed |
| `overrides` | Inject a precomputed value for any node — great for testing downstream logic |
| `PandasDataFrameResult` | Adapter that collects Series outputs into a DataFrame |
| Graph inspection | `list_available_variables`, `what_is_upstream_of`, `what_is_downstream_of` |

> **Tip:** The Driver is the engine — it discovers functions, builds the graph, and executes only what's needed for your requested outputs. Always visualize before you execute.

---
## What's next
**Day 3** → Hamilton decorators: `@config.when` for environment branching, `@tag` for metadata, and `@extract_columns` for splitting DataFrames into named Series.

Mark Day 2 complete in your [tracker](../index.html).